# Supply Chain Analysis

PySpark analysis of the DataCo order-item dataset, validated against three C++ Hadoop Streaming jobs.

## 1. Setup

In [1]:
import getpass
import json
import os
import shutil
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path


from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.classification import DecisionTreeClassifier
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    desc,
    round as spark_round,
    sum as spark_sum,
    when,
)

In [2]:
def find_project_root() -> Path:
    env_root = os.environ.get("BDA_PROJECT_ROOT")
    if env_root:
        return Path(env_root).expanduser().resolve()
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "DataCoSupplyChainDataset.csv").exists():
            return candidate
    return start

In [3]:
PROJECT_ROOT = find_project_root()
DATA_PATH = Path(
    os.environ.get(
        "BDA_DATA_PATH", PROJECT_ROOT / "data" / "DataCoSupplyChainDataset.csv"
    )
).expanduser().resolve()
OUTPUT_DIR = Path(
    os.environ.get("BDA_OUTPUT_DIR", PROJECT_ROOT / "outputs")
).expanduser().resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

USE_HDFS = os.environ.get("BDA_USE_HDFS", "0").lower() in {"1", "true", "yes"}
HDFS_URI = os.environ.get("BDA_HDFS_URI", "hdfs://localhost:9000").rstrip("/")
HDFS_USER = os.environ.get("HDFS_USER", getpass.getuser())
HDFS_INPUT = os.environ.get(
    "BDA_HDFS_INPUT",
    f"{HDFS_URI}/user/{HDFS_USER}/bda/input/DataCoSupplyChainDataset.csv",
)
HDFS_OUTPUT_BASE = os.environ.get(
    "BDA_HDFS_OUTPUT_BASE",
    f"{HDFS_URI}/user/{HDFS_USER}/bda/pyspark/output",
)

In [4]:
def configure_spark_java() -> str:
    override = os.environ.get("BDA_SPARK_JAVA_HOME")
    if override:
        java_home = Path(override).expanduser()
        if (java_home / "bin" / "java").exists():
            os.environ["JAVA_HOME"] = str(java_home)
            return str(java_home)
        raise RuntimeError(f"BDA_SPARK_JAVA_HOME does not contain bin/java: {java_home}")
    java_bin = shutil.which("java")
    if java_bin:
        java_home = Path(java_bin).resolve().parents[1]
        os.environ["JAVA_HOME"] = str(java_home)
        return str(java_home)
    raise RuntimeError(
        "Java was not found on PATH. Install Java 17+ or set BDA_SPARK_JAVA_HOME."
    )


JAVA_HOME = configure_spark_java()
print("Project root :", PROJECT_ROOT)
print("Dataset path :", DATA_PATH)
print("Output dir   :", OUTPUT_DIR)
print("HDFS mode    :", USE_HDFS)
print("Java home    :", JAVA_HOME)

Project root : /home/arsen1c/Documents/Sem_5_Labs/BDA/Project
Dataset path : /home/arsen1c/Documents/Sem_5_Labs/BDA/Project/data/DataCoSupplyChainDataset.csv
Output dir   : /home/arsen1c/Documents/Sem_5_Labs/BDA/Project/outputs
HDFS mode    : False
Java home    : /usr/lib/jvm/java-17-openjdk


In [5]:
spark = (
    SparkSession.builder.master(os.environ.get("SPARK_MASTER", "local[*]"))
    .appName("SupplyChainDeliveryAnalysis")
    .config("spark.sql.shuffle.partitions", os.environ.get("SPARK_SHUFFLE_PARTITIONS", "8"))
    .config("spark.driver.memory", os.environ.get("SPARK_DRIVER_MEMORY", "2g"))
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.pyspark.python", os.environ.get("PYSPARK_PYTHON", sys.executable))
    .config("spark.pyspark.driver.python", os.environ.get("PYSPARK_DRIVER_PYTHON", sys.executable))
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark version:", spark.version)
print("Master       :", spark.sparkContext.master)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/22 23:49:03 WARN Utils: Your hostname, hpomen16-arsen1c, resolves to a loopback address: 127.0.1.1; using 192.168.1.11 instead (on interface wlan0)
26/10/22 23:49:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 4.2.0
Master       : local[*]


## 2. Load data

In [6]:
CSV_OPTIONS = dict(
    header=True,
    inferSchema=True,
    encoding="ISO-8859-1",
    multiLine=True,
    quote='"',
    escape='"',
)

In [7]:
def load_raw_data():
    if USE_HDFS:
        try:
            frame = spark.read.csv(HDFS_INPUT, **CSV_OPTIONS)
            print("Loaded from HDFS:", HDFS_INPUT)
            return frame, HDFS_INPUT
        except Exception as exc:
            print("HDFS read failed; falling back to the local CSV.")
            print("Reason:", exc)

    if not DATA_PATH.exists():
        raise FileNotFoundError(
            f"Dataset not found: {DATA_PATH}\n"
            "Set BDA_DATA_PATH to the CSV path or place the file under data/."
        )

    frame = spark.read.csv(str(DATA_PATH), **CSV_OPTIONS)
    print("Loaded from local file:", DATA_PATH)
    return frame, str(DATA_PATH)

In [8]:
df_raw, input_source = load_raw_data()
df_raw.cache()
print("Rows   :", df_raw.count())
print("Columns:", len(df_raw.columns))

Loaded from local file: /home/arsen1c/Documents/Sem_5_Labs/BDA/Project/data/DataCoSupplyChainDataset.csv


Rows   : 180519
Columns: 53


## 3. Explore

In [9]:
df_raw.printSchema()
df_raw.show(5, truncate=True)

root
 |-- Type: string (nullable = true)
 |-- Days for shipping (real): integer (nullable = true)
 |-- Days for shipment (scheduled): integer (nullable = true)
 |-- Benefit per order: double (nullable = true)
 |-- Sales per customer: double (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: integer (nullable = true)
 |-- Category Id: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Customer City: string (nullable = true)
 |-- Customer Country: string (nullable = true)
 |-- Customer Email: string (nullable = true)
 |-- Customer Fname: string (nullable = true)
 |-- Customer Id: integer (nullable = true)
 |-- Customer Lname: string (nullable = true)
 |-- Customer Password: string (nullable = true)
 |-- Customer Segment: string (nullable = true)
 |-- Customer State: string (nullable = true)
 |-- Customer Street: string (nullable = true)
 |-- Customer Zipcode: integer (nullable = true)
 |-- Department Id: integer (nullable = 

+--------+------------------------+-----------------------------+-----------------+------------------+----------------+------------------+-----------+--------------+-------------+----------------+--------------+--------------+-----------+--------------+-----------------+----------------+--------------+--------------------+----------------+-------------+---------------+-----------+------------+------------+----------+-------------+-----------------+-----------------------+--------+----------------------+-------------------+------------------------+-------------+------------------------+-----------------------+-------------------+------+----------------+----------------------+--------------+---------------+---------------+-------------+---------------+-------------------+-------------------+--------------------+------------+-------------+--------------+--------------------------+--------------+
|    Type|Days for shipping (real)|Days for shipment (scheduled)|Benefit per order|Sales per c

In [10]:
df_raw.select(
    "Order Id",
    "Shipping Mode",
    "Delivery Status",
    "Category Name",
    "Sales",
    "Order Item Total",
).show(10, truncate=False)

+--------+--------------+-----------------+--------------+------+----------------+
|Order Id|Shipping Mode |Delivery Status  |Category Name |Sales |Order Item Total|
+--------+--------------+-----------------+--------------+------+----------------+
|77202   |Standard Class|Advance shipping |Sporting Goods|327.75|314.6400146     |
|75939   |Standard Class|Late delivery    |Sporting Goods|327.75|311.3599854     |
|75938   |Standard Class|Shipping on time |Sporting Goods|327.75|309.7200012     |
|75937   |Standard Class|Advance shipping |Sporting Goods|327.75|304.8099976     |
|75936   |Standard Class|Advance shipping |Sporting Goods|327.75|298.25          |
|75935   |Standard Class|Shipping canceled|Sporting Goods|327.75|294.980011      |
|75934   |First Class   |Late delivery    |Sporting Goods|327.75|288.4200134     |
|75933   |First Class   |Late delivery    |Sporting Goods|327.75|285.1400146     |
|75932   |Second Class  |Late delivery    |Sporting Goods|327.75|278.5899963     |
|759

In [11]:
df_raw.describe(
    "Sales",
    "Order Item Total",
    "Order Item Quantity",
    "Days for shipping (real)",
    "Days for shipment (scheduled)",
    "Order Profit Per Order",
).show()

+-------+------------------+------------------+-------------------+------------------------+-----------------------------+----------------------+
|summary|             Sales|  Order Item Total|Order Item Quantity|Days for shipping (real)|Days for shipment (scheduled)|Order Profit Per Order|
+-------+------------------+------------------+-------------------+------------------------+-----------------------------+----------------------+
|  count|            180519|            180519|             180519|                  180519|                       180519|                180519|
|   mean|203.77209608630992|183.10760850793417|  2.127637533999191|      3.4976539865609713|            2.931846509231715|    21.974988638594276|
| stddev|132.27307749970342|120.04367002812612| 1.4534514814226367|        1.62372182837416|           1.3744492800079673|    104.43352574698609|
|    min|       9.989999771|       7.489999771|                  1|                       0|                            0|  

In [12]:
print("Unique Order Id:", df_raw.select("Order Id").distinct().count())
print("Unique Order Item Id:", df_raw.select("Order Item Id").distinct().count())
df_raw.select("Shipping Mode").distinct().orderBy("Shipping Mode").show()
df_raw.select("Delivery Status").distinct().orderBy("Delivery Status").show()

Unique Order Id: 65752


Unique Order Item Id: 180519


+--------------+
| Shipping Mode|
+--------------+
|   First Class|
|      Same Day|
|  Second Class|
|Standard Class|
+--------------+

+-----------------+
|  Delivery Status|
+-----------------+
| Advance shipping|
|    Late delivery|
|Shipping canceled|
| Shipping on time|
+-----------------+



## 4. Clean

In [13]:
RENAME = {
    "Order Id": "order_id",
    "Order Item Id": "order_item_id",
    "Shipping Mode": "shipping_mode",
    "Delivery Status": "delivery_status",
    "Category Name": "category",
    "Order Item Quantity": "quantity",
    "Sales": "sales",
    "Order Item Total": "order_item_total",
    "Order Profit Per Order": "profit",
    "Days for shipping (real)": "actual_days",
    "Days for shipment (scheduled)": "scheduled_days",
    "Late_delivery_risk": "late_risk",
    "Market": "market",
    "Order Region": "order_region",
    "Customer Segment": "customer_segment",
    "Order Status": "order_status",
    "Product Price": "product_price",
    "Order Item Discount Rate": "discount_rate",
    "Order Item Profit Ratio": "profit_ratio",
    "Type": "payment_type",
    "order date (DateOrders)": "order_date_raw",
    "shipping date (DateOrders)": "shipping_date_raw",
}

In [14]:
raw_count = df_raw.count()
missing = (
    df_raw.select(
        [spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c) for c in df_raw.columns]
    )
    .first()
    .asDict()
)
print("Columns with missing values:")
for name, value in missing.items():
    if value:
        print(f"  {name}: {value:,} ({100 * value / raw_count:.1f}%)")

Columns with missing values:
  Customer Lname: 8 (0.0%)
  Customer Zipcode: 3 (0.0%)
  Order Zipcode: 155,679 (86.2%)
  Product Description: 180,519 (100.0%)


In [15]:
df = df_raw.select([col(old).alias(new) for old, new in RENAME.items()])

df = (
    df.withColumn("order_id", col("order_id").cast("long"))
    .withColumn("order_item_id", col("order_item_id").cast("long"))
    .withColumn("quantity", col("quantity").cast("int"))
    .withColumn("scheduled_days", col("scheduled_days").cast("int"))
    .withColumn("actual_days", col("actual_days").cast("int"))
    .withColumn("late_risk", col("late_risk").cast("int"))
    .withColumn("sales", col("sales").cast("double"))
    .withColumn("order_item_total", col("order_item_total").cast("double"))
    .withColumn("profit", col("profit").cast("double"))
    .withColumn("product_price", col("product_price").cast("double"))
    .withColumn("discount_rate", col("discount_rate").cast("double"))
    .withColumn("profit_ratio", col("profit_ratio").cast("double"))
    .withColumn("order_date", F.to_timestamp("order_date_raw", "M/d/yyyy H:mm"))
    .withColumn("shipping_date", F.to_timestamp("shipping_date_raw", "M/d/yyyy H:mm"))
    .drop("order_date_raw", "shipping_date_raw")
)

In [16]:
for text_col in [
    "shipping_mode",
    "delivery_status",
    "category",
    "market",
    "order_region",
    "customer_segment",
    "order_status",
    "payment_type",
]:
    df = df.withColumn(text_col, F.trim(col(text_col)))

df = df.dropDuplicates(["order_item_id"])
df = df.dropna(
    subset=[
        "order_id",
        "order_item_id",
        "shipping_mode",
        "delivery_status",
        "category",
        "sales",
        "order_item_total",
    ]
)
df = df.fillna({"discount_rate": 0.0, "profit_ratio": 0.0, "quantity": 0})

In [17]:
df = (
    df.withColumn(
        "delivery_group",
        when(col("delivery_status") == "Late delivery", "late")
        .when(
            col("delivery_status").isin("Shipping on time", "Advance shipping"),
            "on_time",
        )
        .when(col("delivery_status") == "Shipping canceled", "canceled")
        .otherwise("other"),
    )
    .withColumn("is_late", when(col("delivery_status") == "Late delivery", 1).otherwise(0))
    .withColumn("delivery_delta_days", col("actual_days") - col("scheduled_days"))
)

In [18]:
# One row per order: sum item measures and take the (consistent) shipping/status
# values once. Category is intentionally not part of order-level features because
# an order can contain items from several categories.
orders = (
    df.groupBy("order_id")
    .agg(
        F.first("shipping_mode", ignorenulls=True).alias("shipping_mode"),
        F.first("delivery_status", ignorenulls=True).alias("delivery_status"),
        F.first("delivery_group", ignorenulls=True).alias("delivery_group"),
        F.first("scheduled_days", ignorenulls=True).alias("scheduled_days"),
        F.first("actual_days", ignorenulls=True).alias("actual_days"),
        F.first("market", ignorenulls=True).alias("market"),
        F.first("order_region", ignorenulls=True).alias("order_region"),
        F.first("customer_segment", ignorenulls=True).alias("customer_segment"),
        F.first("payment_type", ignorenulls=True).alias("payment_type"),
        F.first("order_date", ignorenulls=True).alias("order_date"),
        spark_sum("quantity").alias("total_quantity"),
        spark_sum("sales").alias("total_sales"),
        spark_sum("order_item_total").alias("order_value"),
        spark_sum("profit").alias("total_profit"),
        F.avg("discount_rate").alias("avg_discount_rate"),
    )
    .withColumn("is_late", when(col("delivery_group") == "late", 1).otherwise(0))
    .withColumn("delivery_delta_days", col("actual_days") - col("scheduled_days"))
)

In [19]:
orders.cache()
df.createOrReplaceTempView("delivery_items")
orders.createOrReplaceTempView("delivery_orders")

print("Cleaned order-item rows:", df.count())
print("Order-level rows       :", orders.count())
orders.printSchema()

Cleaned order-item rows: 180519


Order-level rows       : 65752
root
 |-- order_id: long (nullable = true)
 |-- shipping_mode: string (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- delivery_group: string (nullable = true)
 |-- scheduled_days: integer (nullable = true)
 |-- actual_days: integer (nullable = true)
 |-- market: string (nullable = true)
 |-- order_region: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- total_quantity: long (nullable = true)
 |-- total_sales: double (nullable = true)
 |-- order_value: double (nullable = true)
 |-- total_profit: double (nullable = true)
 |-- avg_discount_rate: double (nullable = true)
 |-- is_late: integer (nullable = false)
 |-- delivery_delta_days: integer (nullable = true)



## 5. RDD demo

In [20]:
sc = spark.sparkContext

numbers = sc.parallelize(range(1, 21), 4)
squares = numbers.map(lambda value: value * value)
even_squares = squares.filter(lambda value: value % 2 == 0)

print("RDD partitions       :", numbers.getNumPartitions())
print("map -> squares       :", squares.take(5))
print("filter -> even squares:", even_squares.take(5))
print("even squares count   :", even_squares.count())

RDD partitions       : 4


map -> squares       : [1, 4, 9, 16, 25]
filter -> even squares: [4, 16, 36, 64, 100]


even squares count   : 10


In [21]:
phrase_rdd = sc.parallelize(
    [
        "late delivery standard class",
        "on time delivery same day",
        "late delivery first class",
        "delivery canceled",
    ],
    2,
)
words = phrase_rdd.flatMap(lambda line: line.lower().split())
print("flatMap sample       :", words.take(10))
print("distinct word count  :", words.distinct().count())
print("distinct sample      :", words.distinct().take(10))

flatMap sample       : ['late', 'delivery', 'standard', 'class', 'on', 'time', 'delivery', 'same', 'day', 'late']


distinct word count  : 10


distinct sample      : ['standard', 'class', 'day', 'first', 'canceled', 'late', 'delivery', 'on', 'time', 'same']


In [22]:
try:
    lines = sc.textFile(input_source)
    print("textFile line count  :", lines.count())
    print("textFile first line  :", lines.first()[:120])
except Exception as exc:
    print("textFile demo skipped:", exc)

textFile line count  : 180520
textFile first line  : Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_de


## 6. Shipping summary

In [23]:
mode_summary = (
    orders.groupBy("shipping_mode")
    .agg(
        count("*").alias("total_orders"),
        spark_sum(when(col("delivery_group") == "late", 1).otherwise(0)).alias("late_orders"),
        spark_sum(when(col("delivery_group") == "on_time", 1).otherwise(0)).alias("on_time_orders"),
        spark_sum(when(col("delivery_group") == "canceled", 1).otherwise(0)).alias("canceled_orders"),
        spark_round(F.avg("actual_days"), 2).alias("avg_actual_days"),
        spark_round(F.avg("scheduled_days"), 2).alias("avg_scheduled_days"),
        spark_round(F.avg("order_value"), 2).alias("avg_order_value_proxy"),
        spark_round(spark_sum("order_value"), 2).alias("total_order_value_proxy"),
        spark_round(F.avg("total_profit"), 2).alias("avg_profit_proxy"),
    )
    .withColumn("active_orders", col("late_orders") + col("on_time_orders"))
    .withColumn(
        "late_rate_pct",
        spark_round(100 * col("late_orders") / col("total_orders"), 2),
    )
    .withColumn(
        "late_rate_active_pct",
        spark_round(100 * col("late_orders") / col("active_orders"), 2),
    )
    .withColumnRenamed("avg_order_value_proxy", "avg_order_value_proxy")
    .orderBy(desc("late_rate_pct"))
)

In [24]:
mode_summary.show(truncate=False)

active_orders = orders.where(col("delivery_group") != "canceled")
print("Active (late + on-time) orders:", active_orders.count())

+--------------+------------+-----------+--------------+---------------+---------------+------------------+---------------------+-----------------------+----------------+-------------+-------------+--------------------+
|shipping_mode |total_orders|late_orders|on_time_orders|canceled_orders|avg_actual_days|avg_scheduled_days|avg_order_value_proxy|total_order_value_proxy|avg_profit_proxy|active_orders|late_rate_pct|late_rate_active_pct|
+--------------+------------+-----------+--------------+---------------+---------------+------------------+---------------------+-----------------------+----------------+-------------+-------------+--------------------+
|First Class   |10079       |9602       |0             |477            |2.0            |1.0               |506.05               |5100451.07             |63.81           |9602         |95.27        |100.0               |
|Second Class  |12778       |9803       |2453          |522            |4.0            |2.0               |502.15       

## 7. Analysis

### Delivery status

In [25]:
delivery_counts = orders.groupBy("delivery_group").count().orderBy(desc("count"))
delivery_counts.show()

total_orders = orders.count()
late_orders = orders.filter(col("delivery_group") == "late").count()
active_count = orders.filter(col("delivery_group").isin("late", "on_time")).count()

print(f"Total orders                 : {total_orders:,}")
print(f"Late orders                  : {late_orders:,} ({100 * late_orders / total_orders:.2f}% of all orders)")
print(f"Late rate among active orders: {100 * late_orders / active_count:.2f}%")

+--------------+-----+
|delivery_group|count|
+--------------+-----+
|          late|36048|
|       on_time|26849|
|      canceled| 2855|
+--------------+-----+



Total orders                 : 65,752
Late orders                  : 36,048 (54.82% of all orders)
Late rate among active orders: 57.31%


In [26]:
orders.groupBy("delivery_group").agg(
    spark_round(F.avg("actual_days"), 2).alias("avg_actual_days"),
    spark_round(F.avg("scheduled_days"), 2).alias("avg_scheduled_days"),
    spark_round(F.avg("delivery_delta_days"), 2).alias("avg_delta_days"),
).orderBy("delivery_group").show()

+--------------+---------------+------------------+--------------+
|delivery_group|avg_actual_days|avg_scheduled_days|avg_delta_days|
+--------------+---------------+------------------+--------------+
|      canceled|           3.49|               2.9|          0.58|
|          late|           4.09|              2.47|          1.62|
|       on_time|           2.71|              3.56|         -0.85|
+--------------+---------------+------------------+--------------+



### Shipping mode

In [27]:
mode_summary.select(
    "shipping_mode",
    "total_orders",
    "late_orders",
    "on_time_orders",
    "canceled_orders",
    "late_rate_pct",
    "late_rate_active_pct",
    "avg_actual_days",
    "avg_scheduled_days",
).show(truncate=False)

+--------------+------------+-----------+--------------+---------------+-------------+--------------------+---------------+------------------+
|shipping_mode |total_orders|late_orders|on_time_orders|canceled_orders|late_rate_pct|late_rate_active_pct|avg_actual_days|avg_scheduled_days|
+--------------+------------+-----------+--------------+---------------+-------------+--------------------+---------------+------------------+
|First Class   |10079       |9602       |0             |477            |95.27        |100.0               |2.0            |1.0               |
|Second Class  |12778       |9803       |2453          |522            |76.72        |79.99               |4.0            |2.0               |
|Same Day      |3571        |1648       |1759          |164            |46.15        |48.37               |0.48           |0.0               |
|Standard Class|39324       |14995      |22637         |1692           |38.13        |39.85               |4.0            |4.0               |

In [28]:
orders.groupBy("shipping_mode").pivot("delivery_group").count().na.fill(0).orderBy(
    "shipping_mode"
).show()

+--------------+--------+-----+-------+
| shipping_mode|canceled| late|on_time|
+--------------+--------+-----+-------+
|   First Class|     477| 9602|      0|
|      Same Day|     164| 1648|   1759|
|  Second Class|     522| 9803|   2453|
|Standard Class|    1692|14995|  22637|
+--------------+--------+-----+-------+



### Monetary proxies

No shipping-cost column; order value and profit are proxies.

In [29]:
mode_summary.select(
    "shipping_mode",
    "avg_order_value_proxy",
    "total_order_value_proxy",
    "avg_profit_proxy",
    "avg_actual_days",
    "avg_scheduled_days",
).orderBy("shipping_mode").show(truncate=False)

print(
    "Reminder: 'order value' is the sum of Order Item Total per order; "
    "it is not a shipping charge."
)

+--------------+---------------------+-----------------------+----------------+---------------+------------------+
|shipping_mode |avg_order_value_proxy|total_order_value_proxy|avg_profit_proxy|avg_actual_days|avg_scheduled_days|
+--------------+---------------------+-----------------------+----------------+---------------+------------------+
|First Class   |506.05               |5100451.07             |63.81           |2.0            |1.0               |
|Same Day      |488.72               |1745201.53             |56.85           |0.48           |0.0               |
|Second Class  |502.15               |6416528.54             |58.72           |4.0            |2.0               |
|Standard Class|503.31               |1.979222124E7          |60.28           |4.0            |4.0               |
+--------------+---------------------+-----------------------+----------------+---------------+------------------+

Reminder: 'order value' is the sum of Order Item Total per order; it is not a s

### Categories

Item-level demand proxy; no inventory-stock data.

In [30]:
category_summary = (
    df.groupBy("category")
    .agg(
        count("*").alias("order_items"),
        countDistinct("order_id").alias("orders"),
        spark_sum(when(col("delivery_group") == "late", 1).otherwise(0)).alias("late_items"),
        spark_round(spark_sum("sales"), 2).alias("total_sales"),
        spark_round(F.avg("order_item_total"), 2).alias("avg_item_value"),
    )
    .withColumn(
        "late_rate_pct",
        spark_round(100 * col("late_items") / col("order_items"), 2),
    )
    .orderBy(desc("orders"))
)

In [31]:
category_summary.show(10, truncate=False)

+--------------------+-----------+------+----------+-----------+--------------+-------------+
|category            |order_items|orders|late_items|total_sales|avg_item_value|late_rate_pct|
+--------------------+-----------+------+----------+-----------+--------------+-------------+
|Cleats              |24551      |20386 |13496     |4431942.78 |162.23        |54.97        |
|Men's Footwear      |22246      |18783 |12121     |2891757.66 |116.81        |54.49        |
|Women's Apparel     |21035      |17869 |11476     |3147800.0  |134.48        |54.56        |
|Indoor/Outdoor Games|19298      |16623 |10565     |2888993.91 |134.55        |54.75        |
|Fishing             |17325      |15164 |9516      |6929653.69 |359.42        |54.93        |
|Water Sports        |15540      |13758 |8517      |3113844.68 |180.05        |54.81        |
|Camping & Hiking    |13729      |12299 |7487      |4118425.57 |269.56        |54.53        |
|Cardio Equipment    |12487      |11355 |6805      |3694843.

## 8. Spark SQL

In [32]:
orders.createOrReplaceTempView("delivery_orders")

spark.sql(
    """
    SELECT
        shipping_mode,
        COUNT(*) AS orders,
        SUM(CASE WHEN is_late = 1 THEN 1 ELSE 0 END) AS late_orders,
        ROUND(100.0 * SUM(CASE WHEN is_late = 1 THEN 1 ELSE 0 END) / COUNT(*), 2) AS late_rate_pct
    FROM delivery_orders
    WHERE delivery_group <> 'canceled'
    GROUP BY shipping_mode
    ORDER BY late_rate_pct DESC
    """
).show(truncate=False)

+--------------+------+-----------+-------------+
|shipping_mode |orders|late_orders|late_rate_pct|
+--------------+------+-----------+-------------+
|First Class   |9602  |9602       |100.00       |
|Second Class  |12256 |9803       |79.99        |
|Same Day      |3407  |1648       |48.37        |
|Standard Class|37632 |14995      |39.85        |
+--------------+------+-----------+-------------+



## 9. Hadoop outputs

Optional: loads the local result bundle when available.

In [33]:
from pyspark.sql import types as T

HADOOP_RESULTS_DIR = Path(
    os.environ.get("BDA_HADOOP_RESULTS", PROJECT_ROOT / "results" / "hadoop")
).expanduser().resolve()
HADOOP_RUN_ID = os.environ.get("BDA_HADOOP_RUN_ID", "").strip()
HADOOP_HDFS_ANALYTICS_BASE = os.environ.get(
    "BDA_HDFS_ANALYTICS_BASE",
    f"{HDFS_URI}/user/{HDFS_USER}/bda/analytics",
).rstrip("/")
HADOOP_HDFS_RESULTS_BASE = HADOOP_HDFS_ANALYTICS_BASE + "/output"
HADOOP_HDFS_STREAMING_BASE = os.environ.get(
    "BDA_HDFS_BASE", f"{HDFS_URI}/user/{HDFS_USER}/bda/streaming"
).rstrip("/")

In [34]:
def hadoop_local_path(job_name: str):
    if not HADOOP_RESULTS_DIR.exists():
        return None
    if HADOOP_RUN_ID:
        candidate = HADOOP_RESULTS_DIR / HADOOP_RUN_ID / f"{job_name}.tsv"
        return candidate if candidate.exists() else None
    candidates = sorted(
        HADOOP_RESULTS_DIR.glob(f"*/{job_name}.tsv"),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    return candidates[0] if candidates else None

In [35]:
def read_hadoop_result(job_name: str, schema: T.StructType):
    local_path = hadoop_local_path(job_name)
    if local_path is not None:
        frame = (
            spark.read.option("sep", "\t")
            .option("header", "false")
            .schema(schema)
            .csv(str(local_path))
        )
        print(f"Loaded Hadoop result: {local_path}")
        return frame

    if USE_HDFS and HADOOP_RUN_ID:
        if job_name == "delivery-mode":
            hdfs_path = f"{HADOOP_HDFS_STREAMING_BASE}/output/{HADOOP_RUN_ID}"
        else:
            hdfs_path = f"{HADOOP_HDFS_RESULTS_BASE}/{HADOOP_RUN_ID}/{job_name}"
        try:
            frame = (
                spark.read.option("sep", "\t")
                .option("header", "false")
                .schema(schema)
                .csv(hdfs_path)
            )
            print(f"Loaded Hadoop result: {hdfs_path}")
            return frame
        except Exception as exc:
            print(f"HDFS Hadoop-result read failed: {hdfs_path}")
            print("Reason:", exc)

    print(
        f"No Hadoop result found for {job_name!r}. Run the C++ jobs first: "
        "bash scripts/run_all_hadoop_jobs.sh"
    )
    return None

In [36]:
DELIVERY_HADOOP_SCHEMA = T.StructType(
    [
        T.StructField("shipping_mode", T.StringType()),
        T.StructField("status", T.StringType()),
        T.StructField("distinct_orders", T.LongType()),
    ]
)

In [37]:
SHIPPING_HADOOP_SCHEMA = T.StructType(
    [
        T.StructField("shipping_mode", T.StringType()),
        T.StructField("records", T.LongType()),
        T.StructField("distinct_orders", T.LongType()),
        T.StructField("late_records", T.LongType()),
        T.StructField("on_time_records", T.LongType()),
        T.StructField("canceled_records", T.LongType()),
        T.StructField("other_records", T.LongType()),
        T.StructField("avg_actual_days", T.DoubleType()),
        T.StructField("avg_scheduled_days", T.DoubleType()),
        T.StructField("avg_delay_days", T.DoubleType()),
        T.StructField("total_sales", T.DoubleType()),
        T.StructField("avg_sales_per_order", T.DoubleType()),
    ]
)

In [38]:
CATEGORY_HADOOP_SCHEMA = T.StructType(
    [
        T.StructField("category", T.StringType()),
        T.StructField("records", T.LongType()),
        T.StructField("distinct_orders", T.LongType()),
        T.StructField("total_quantity", T.LongType()),
        T.StructField("units_per_order", T.DoubleType()),
        T.StructField("total_sales", T.DoubleType()),
        T.StructField("avg_sales_per_item", T.DoubleType()),
        T.StructField("late_records", T.LongType()),
        T.StructField("late_rate_pct", T.DoubleType()),
    ]
)

In [39]:
delivery_hadoop = read_hadoop_result("delivery-mode", DELIVERY_HADOOP_SCHEMA)
shipping_hadoop = read_hadoop_result("shipping-performance", SHIPPING_HADOOP_SCHEMA)
category_hadoop = read_hadoop_result("category-demand", CATEGORY_HADOOP_SCHEMA)

Loaded Hadoop result: /home/arsen1c/Documents/Sem_5_Labs/BDA/Project/results/hadoop/retry-20261022T175034Z/delivery-mode.tsv
Loaded Hadoop result: /home/arsen1c/Documents/Sem_5_Labs/BDA/Project/results/hadoop/retry-20261022T175034Z/shipping-performance.tsv
Loaded Hadoop result: /home/arsen1c/Documents/Sem_5_Labs/BDA/Project/results/hadoop/retry-20261022T175034Z/category-demand.tsv


### Job 1: delivery status

In [40]:
if delivery_hadoop is not None:
    print("\nHadoop Job 1 — delivery status")
    delivery_hadoop.orderBy("shipping_mode", "status").show(20, truncate=False)


Hadoop Job 1 — delivery status
+--------------+--------+---------------+
|shipping_mode |status  |distinct_orders|
+--------------+--------+---------------+
|First Class   |canceled|477            |
|First Class   |late    |9602           |
|Same Day      |canceled|164            |
|Same Day      |late    |1648           |
|Same Day      |on_time |1759           |
|Second Class  |canceled|522            |
|Second Class  |late    |9803           |
|Second Class  |on_time |2453           |
|Standard Class|canceled|1692           |
|Standard Class|late    |14995          |
|Standard Class|on_time |22637          |
+--------------+--------+---------------+



In [41]:
if delivery_hadoop is not None:
    delivery_spark = (
        orders.groupBy("shipping_mode", F.col("delivery_group").alias("status"))
        .agg(F.count("*").alias("spark_orders"))
    )

In [42]:
if delivery_hadoop is not None:
    delivery_compare = (
        delivery_hadoop.join(
            delivery_spark, ["shipping_mode", "status"], "full_outer"
        )
        .withColumn(
            "hadoop_orders", F.coalesce(F.col("distinct_orders"), F.lit(0)).cast("long")
        )
        .withColumn(
            "spark_orders", F.coalesce(F.col("spark_orders"), F.lit(0)).cast("long")
        )
        .withColumn("difference", F.col("hadoop_orders") - F.col("spark_orders"))
        .select(
            "shipping_mode",
            "status",
            "hadoop_orders",
            "spark_orders",
            "difference",
        )
        .orderBy("shipping_mode", "status")
    )

In [43]:
if delivery_hadoop is not None:
    print("Distinct-order comparison (Job 1 vs PySpark):")
    delivery_compare.show(20, truncate=False)

Distinct-order comparison (Job 1 vs PySpark):


+--------------+--------+-------------+------------+----------+
|shipping_mode |status  |hadoop_orders|spark_orders|difference|
+--------------+--------+-------------+------------+----------+
|First Class   |canceled|477          |477         |0         |
|First Class   |late    |9602         |9602        |0         |
|Same Day      |canceled|164          |164         |0         |
|Same Day      |late    |1648         |1648        |0         |
|Same Day      |on_time |1759         |1759        |0         |
|Second Class  |canceled|522          |522         |0         |
|Second Class  |late    |9803         |9803        |0         |
|Second Class  |on_time |2453         |2453        |0         |
|Standard Class|canceled|1692         |1692        |0         |
|Standard Class|late    |14995        |14995       |0         |
|Standard Class|on_time |22637        |22637       |0         |
+--------------+--------+-------------+------------+----------+



### Job 2: shipping performance

In [44]:
if shipping_hadoop is not None:
    print("\nHadoop Job 2 — shipping performance (item-row metrics)")
    shipping_hadoop.select(
        "shipping_mode",
        "records",
        "distinct_orders",
        "late_records",
        "on_time_records",
        "canceled_records",
        "avg_actual_days",
        "avg_scheduled_days",
        "avg_delay_days",
        "avg_sales_per_order",
    ).orderBy("shipping_mode").show(truncate=False)


Hadoop Job 2 — shipping performance (item-row metrics)
+--------------+-------+---------------+------------+---------------+----------------+---------------+------------------+--------------+-------------------+
|shipping_mode |records|distinct_orders|late_records|on_time_records|canceled_records|avg_actual_days|avg_scheduled_days|avg_delay_days|avg_sales_per_order|
+--------------+-------+---------------+------------+---------------+----------------+---------------+------------------+--------------+-------------------+
|First Class   |27814  |10079          |26513       |0              |1301            |2.0            |1.0               |1.0           |562.99             |
|Same Day      |9737   |3571           |4454        |4839           |444             |0.48           |0.0               |0.48          |543.97             |
|Second Class  |35216  |12778          |26987       |6819           |1410            |3.99           |2.0               |1.99          |559.2              |
|S

In [45]:
if shipping_hadoop is not None:
    shipping_compare = (
        shipping_hadoop.select(
            "shipping_mode", F.col("distinct_orders").alias("hadoop_orders")
        )
        .join(
            mode_summary.select(
                "shipping_mode", F.col("total_orders").alias("spark_orders")
            ),
            "shipping_mode",
            "full_outer",
        )
        .withColumn(
            "hadoop_orders", F.coalesce(F.col("hadoop_orders"), F.lit(0)).cast("long")
        )
        .withColumn(
            "spark_orders", F.coalesce(F.col("spark_orders"), F.lit(0)).cast("long")
        )
        .withColumn("difference", F.col("hadoop_orders") - F.col("spark_orders"))
        .orderBy("shipping_mode")
    )

In [46]:
if shipping_hadoop is not None:
    print("Distinct-order comparison (Job 2 vs PySpark):")
    shipping_compare.show(truncate=False)
    print(
        "Job 2 records are order-item rows; do not compare late_records with "
        "order-level late_orders."
    )

Distinct-order comparison (Job 2 vs PySpark):
+--------------+-------------+------------+----------+
|shipping_mode |hadoop_orders|spark_orders|difference|
+--------------+-------------+------------+----------+
|First Class   |10079        |10079       |0         |
|Same Day      |3571         |3571        |0         |
|Second Class  |12778        |12778       |0         |
|Standard Class|39324        |39324       |0         |
+--------------+-------------+------------+----------+

Job 2 records are order-item rows; do not compare late_records with order-level late_orders.


### Job 3: category demand

In [47]:
if category_hadoop is not None:
    print("\nHadoop Job 3 — category demand (top 10 by quantity)")
    category_hadoop.select(
        "category",
        "records",
        "distinct_orders",
        "total_quantity",
        "units_per_order",
        "total_sales",
        "late_records",
        "late_rate_pct",
    ).orderBy(F.desc("total_quantity")).show(10, truncate=False)


Hadoop Job 3 — category demand (top 10 by quantity)


+--------------------+-------+---------------+--------------+---------------+-----------+------------+-------------+
|category            |records|distinct_orders|total_quantity|units_per_order|total_sales|late_records|late_rate_pct|
+--------------------+-------+---------------+--------------+---------------+-----------+------------+-------------+
|Cleats              |24551  |20386          |73734         |3.62           |4431942.78 |13496       |54.97        |
|Women's Apparel     |21035  |17869          |62956         |3.52           |3147800.0  |11476       |54.56        |
|Indoor/Outdoor Games|19298  |16623          |57803         |3.48           |2888993.91 |10565       |54.75        |
|Cardio Equipment    |12487  |11355          |37587         |3.31           |3694843.2  |6805        |54.5         |
|Shop By Sport       |10984  |10136          |32726         |3.23           |1309522.04 |6058        |55.15        |
|Men's Footwear      |22246  |18783          |22246         |1.1

In [48]:
if category_hadoop is not None:
    category_compare = (
        category_hadoop.select(
            "category",
            F.col("records").alias("hadoop_items"),
            F.col("distinct_orders").alias("hadoop_orders"),
            F.col("total_sales").alias("hadoop_sales"),
        )
        .join(
            category_summary.select(
                "category",
                F.col("order_items").alias("spark_items"),
                F.col("orders").alias("spark_orders"),
                F.col("total_sales").alias("spark_sales"),
            ),
            "category",
            "full_outer",
        )
        .withColumn("item_difference", F.col("hadoop_items") - F.col("spark_items"))
        .withColumn("order_difference", F.col("hadoop_orders") - F.col("spark_orders"))
        .withColumn(
            "sales_difference",
            F.round(F.col("hadoop_sales") - F.col("spark_sales"), 2),
        )
        .orderBy(F.desc("hadoop_items"))
    )


In [49]:
if category_hadoop is not None:
    print("Category comparison (Job 3 vs PySpark; first 10 rows):")
    category_compare.show(10, truncate=False)

Category comparison (Job 3 vs PySpark; first 10 rows):


+--------------------+------------+-------------+------------+-----------+------------+-----------+---------------+----------------+----------------+
|category            |hadoop_items|hadoop_orders|hadoop_sales|spark_items|spark_orders|spark_sales|item_difference|order_difference|sales_difference|
+--------------------+------------+-------------+------------+-----------+------------+-----------+---------------+----------------+----------------+
|Cleats              |24551       |20386        |4431942.78  |24551      |20386       |4431942.78 |0              |0               |0.0             |
|Men's Footwear      |22246       |18783        |2891757.66  |22246      |18783       |2891757.66 |0              |0               |0.0             |
|Women's Apparel     |21035       |17869        |3147800.0   |21035      |17869       |3147800.0  |0              |0               |0.0             |
|Indoor/Outdoor Games|19298       |16623        |2888993.91  |19298      |16623       |2888993.91 |0

In [50]:
if all(
    frame is None
    for frame in (delivery_hadoop, shipping_hadoop, category_hadoop)
):
    print(
        "No local Hadoop bundle found under "
        f"{HADOOP_RESULTS_DIR}. The PySpark analysis above is unaffected."
    )

## 10. Save results

In [51]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]

In [52]:
def save_parquet(frame, name):
    if USE_HDFS:
        target = f"{HDFS_OUTPUT_BASE}/{RUN_ID}/{name}"
        try:
            frame.coalesce(1).write.mode("errorifexists").parquet(target)
            print("Saved to HDFS:", target)
            return target
        except Exception as exc:
            print("HDFS write failed; using the local fallback.")
            print("Reason:", exc)

    target = OUTPUT_DIR / "pyspark" / RUN_ID / name
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        raise FileExistsError(f"Output already exists: {target}")
    frame.coalesce(1).write.mode("errorifexists").parquet(str(target))
    print("Saved locally:", target)
    return str(target)

In [53]:
mode_output = save_parquet(mode_summary, "shipping_mode_summary")
category_output = save_parquet(category_summary, "category_summary")
print("Run id:", RUN_ID)

Saved locally: /home/arsen1c/Documents/Sem_5_Labs/BDA/Project/outputs/pyspark/20261022T181926Z_746773/shipping_mode_summary


Saved locally: /home/arsen1c/Documents/Sem_5_Labs/BDA/Project/outputs/pyspark/20261022T181926Z_746773/category_summary
Run id: 20261022T181926Z_746773


## 11. Late-delivery model

Canceled orders are excluded; post-delivery fields are not features.

In [54]:
ml_df = (
    orders.filter(col("delivery_group").isin("late", "on_time"))
    .select(
        "order_id",
        "is_late",
        "shipping_mode",
        "scheduled_days",
        "total_quantity",
        "total_sales",
        "avg_discount_rate",
        "market",
        "order_region",
        "customer_segment",
        "payment_type",
    )
)

In [55]:
ml_df = (
    ml_df.fillna(
        {
            "scheduled_days": 0,
            "total_quantity": 0,
            "total_sales": 0.0,
            "avg_discount_rate": 0.0,
        }
    )
    .fillna(
        "Unknown",
        subset=[
            "shipping_mode",
            "market",
            "order_region",
            "customer_segment",
            "payment_type",
        ],
    )
)

In [56]:
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)
print("Training rows:", train_df.count())
print("Test rows    :", test_df.count())
print("Class balance in training data:")
train_df.groupBy("is_late").count().orderBy("is_late").show()

Training rows: 50533
Test rows    : 12364
Class balance in training data:


+-------+-----+
|is_late|count|
+-------+-----+
|      0|21575|
|      1|28958|
+-------+-----+



In [57]:
categorical_cols = [
    "shipping_mode",
    "market",
    "order_region",
    "customer_segment",
    "payment_type",
]
numeric_cols = [
    "scheduled_days",
    "total_quantity",
    "total_sales",
    "avg_discount_rate",
]
indexed_cols = [f"{name}_idx" for name in categorical_cols]
indexers = [
    StringIndexer(inputCol=name, outputCol=f"{name}_idx", handleInvalid="keep")
    for name in categorical_cols
]

In [58]:
assembler = VectorAssembler(
    inputCols=numeric_cols + indexed_cols,
    outputCol="features",
    handleInvalid="keep",
)
classifier = DecisionTreeClassifier(
    labelCol="is_late",
    featuresCol="features",
    maxDepth=5,
    minInstancesPerNode=20,
    seed=42,
)
pipeline = Pipeline(stages=indexers + [assembler, classifier])

In [59]:
model = pipeline.fit(train_df)
predictions = model.transform(test_df)

In [60]:
accuracy = MulticlassClassificationEvaluator(
    labelCol="is_late", predictionCol="prediction", metricName="accuracy"
).evaluate(predictions)
f1 = MulticlassClassificationEvaluator(
    labelCol="is_late", predictionCol="prediction", metricName="f1"
).evaluate(predictions)
auc = BinaryClassificationEvaluator(
    labelCol="is_late", rawPredictionCol="rawPrediction", metricName="areaUnderROC"
).evaluate(predictions)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 score: {f1:.4f}")
print(f"AUC     : {auc:.4f}")

Accuracy: 0.6915
F1 score: 0.6870
AUC     : 0.7130


In [61]:
print("Confusion matrix (label, prediction, count):")
predictions.groupBy("is_late", "prediction").count().orderBy(
    "is_late", "prediction"
).show()

Confusion matrix (label, prediction, count):


+-------+----------+-----+
|is_late|prediction|count|
+-------+----------+-----+
|      0|       0.0| 4696|
|      0|       1.0|  578|
|      1|       0.0| 3236|
|      1|       1.0| 3854|
+-------+----------+-----+



In [62]:
MODEL_DIR = OUTPUT_DIR / "model"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
metrics = {
    "trained": True,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "train_rows": train_df.count(),
    "test_rows": test_df.count(),
    "accuracy": round(float(accuracy), 4),
    "f1": round(float(f1), 4),
    "auc": round(float(auc), 4),
    "label": "is_late",
    "features": numeric_cols + indexed_cols,
    "excluded_for_leakage": [
        "Delivery Status",
        "Late_delivery_risk",
        "Days for shipping (real)",
        "shipping date (DateOrders)",
        "Order Status",
        "profit fields",
    ],
}

In [63]:
sample_predictions = (
    predictions.select(
        "order_id",
        "is_late",
        "prediction",
        spark_round(vector_to_array(col("probability"))[1], 4).alias(
            "late_probability"
        ),
    )
    .limit(20)
    .toPandas()
)
sample_predictions.to_csv(MODEL_DIR / "predictions_sample.csv", index=False)

model_run_id = (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    + "_"
    + uuid.uuid4().hex[:6]
)
model_path = MODEL_DIR / f"late_delivery_model_{model_run_id}"
model.write().save(str(model_path))
metrics["model_path"] = str(model_path)

with open(MODEL_DIR / "metrics.json", "w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)

print("Saved model artefacts to:", MODEL_DIR)
print(json.dumps(metrics, indent=2))

Saved model artefacts to: /home/arsen1c/Documents/Sem_5_Labs/BDA/Project/outputs/model
{
  "trained": true,
  "trained_at_utc": "2026-10-22T18:19:32.543286+00:00",
  "train_rows": 50533,
  "test_rows": 12364,
  "accuracy": 0.6915,
  "f1": 0.687,
  "auc": 0.713,
  "label": "is_late",
  "features": [
    "scheduled_days",
    "total_quantity",
    "total_sales",
    "avg_discount_rate",
    "shipping_mode_idx",
    "market_idx",
    "order_region_idx",
    "customer_segment_idx",
    "payment_type_idx"
  ],
  "excluded_for_leakage": [
    "Delivery Status",
    "Late_delivery_risk",
    "Days for shipping (real)",
    "shipping date (DateOrders)",
    "Order Status",
    "profit fields"
  ],
  "model_path": "/home/arsen1c/Documents/Sem_5_Labs/BDA/Project/outputs/model/late_delivery_model_20261022T181933Z_567ffb"
}


## 12. Finish

In [64]:
# spark.stop()
print("Notebook finished. Spark session left running for interactive use.")

Notebook finished. Spark session left running for interactive use.
